# Phase 4C.1 Lab - Neural Pricing Boundary

Mục tiêu: hiểu formatter, neural adapter boundary, và fallback khi thiếu
weights/dependencies.

Default expected output: real pricing tests pass/skipped without model calls.

Safety: neural smoke chỉ chạy khi `ENABLE_REAL_MODEL_CALLS=true` và
`PRICER_NEURAL_WEIGHTS_PATH` tồn tại.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def find_v3_root(start: str | None = None) -> Path:
    path = Path(start or os.getcwd()).resolve()
    for candidate in (path, *path.parents):
        if candidate.name == "shopping_assistant_v3" and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the tech2ai/shopping_assistant_v3 tree.")


V3_ROOT = find_v3_root()
os.chdir(V3_ROOT)
if str(V3_ROOT) not in sys.path:
    sys.path.insert(0, str(V3_ROOT))


def run(command: list[str], timeout: int = 120) -> subprocess.CompletedProcess[str] | None:
    print("$ " + " ".join(command))
    try:
        result = subprocess.run(
            command,
            cwd=V3_ROOT,
            text=True,
            capture_output=True,
            timeout=timeout,
        )
    except subprocess.TimeoutExpired:
        print(f"Command timed out after {timeout} seconds.")
        return None

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print(f"exit_code={result.returncode}")
    return result


print(f"V3_ROOT={V3_ROOT}")
print("Default safety flags:")
for name in ("ENABLE_REAL_SEARCH", "ENABLE_REAL_MODEL_CALLS", "ENABLE_AGENTS_SDK"):
    print(f"{name}={os.getenv(name, '<unset>')}")


## 1. Chạy default real-pricing boundary tests

Command này không cần neural weights.


In [ ]:
run(["uv", "run", "pytest", "tests/test_real_pricing.py", "-q", "--tb=short"], timeout=180)


## 2. Demo formatter + fallback real estimator

Expected: nếu thiếu real model config, result dùng fallback markup và warnings.


In [ ]:
from backend.tools.deal_search.schemas import ProductCandidate
from backend.tools.price_estimator.real_estimator import estimate_price_real

product = ProductCandidate(
    source="Amazon",
    title="Example Gaming Laptop 16GB RAM RTX GPU",
    brand="Example",
    sale_price_usd=799.99,
    url="https://www.amazon.com/example",
    features="16GB RAM, RTX GPU, 1TB SSD",
)
estimate = estimate_price_real(product)
print(estimate.model_dump())


## 3. Opt-in neural smoke cell

Cell này chạy neural smoke test chỉ khi env đã đủ. Không commit weights `.pth`.


In [ ]:
weights_path = os.getenv("PRICER_NEURAL_WEIGHTS_PATH", "")
if (
    os.getenv("ENABLE_REAL_MODEL_CALLS", "").strip().lower() == "true"
    and weights_path
    and Path(weights_path).exists()
):
    run(["uv", "run", "pytest", "tests/test_real_pricing_neural.py", "-q", "--tb=short"], timeout=240)
else:
    print("Skipped neural smoke. Requires ENABLE_REAL_MODEL_CALLS=true and PRICER_NEURAL_WEIGHTS_PATH pointing to an existing weights file.")


## 4. Cách đọc kết quả

- Fallback warnings là expected nếu thiếu weights/deps.
- Neural smoke pass chứng minh local PyTorch model load được.
- Phase này chưa có Frontier/Specialist full ensemble.
